# Comparación offline de SigLIP y SigLIP2

Se comparan los checkpoints base de resolución fija 224 bajo el mismo protocolo, con idénticas imágenes, consultas, preprocesado y métricas. El texto de consulta forma parte de la entrada del modelo, por lo que se controla su idioma, longitud, formulación y alcance semántico. El conjunto propio actúa como verificación funcional; SUN RGB-D es el conjunto discriminativo. Los intervalos se obtienen mediante bootstrap por consulta.

In [1]:
from pathlib import Path
import sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'semantic_navigation_ws' / 'src').is_dir())
sys.path.insert(0, str(repo / 'experiments' / 'shared'))
import pandas as pd
from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from offline_benchmarks import create_vlm_figures, run_vlm_benchmark
ctx = bootstrap_offline()
print(f"Dispositivo: {ctx['device']}")
display(pd.DataFrame([{'modelo': key, 'checkpoint': value}
                      for key, value in ctx['config']['models']['siglip']['variants'].items()]))

/home/junior/visual_semantic_navigation/.venv-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda


,modelo,checkpoint
0,siglip_v1,google/siglip-base-patch16-224
1,siglip_v2,google/siglip2-base-patch16-224


## Ejecución y resultados numéricos

In [2]:
results = run_vlm_benchmark(ctx)
cases = results['cases']
summary = results['summary']
display(summary)
display(results['summary_by_query_scope'])
display(results['paired_differences'])
display(results['threshold_diagnostics'])
display(results['model_costs'])

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 5469.89it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|█████████▉| 407/408 [00:00<00:00, 4067.23it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 4024.73it/s]

,dataset_id,method,query_type,language,n_queries,n_positive,n_negative,recall_at_1,recall_at_1_ci_low,recall_at_1_ci_high,...,recall_at_5,recall_at_5_ci_low,recall_at_5_ci_high,mean_reciprocal_rank,mean_reciprocal_rank_ci_low,mean_reciprocal_rank_ci_high,negative_rejection_rate,negative_rejection_rate_ci_low,negative_rejection_rate_ci_high,mean_retrieval_latency_ms
0,siglip_rooms,random_baseline,attribute,es,1,1,0,0.000000,NaN,NaN,...,0.000000,NaN,NaN,0.166667,NaN,NaN,NaN,NaN,NaN,0.245775
1,siglip_rooms,random_baseline,functional,en,1,1,0,0.000000,NaN,NaN,...,1.000000,NaN,NaN,0.500000,NaN,NaN,NaN,NaN,NaN,0.241864
2,siglip_rooms,random_baseline,functional,es,1,1,0,1.000000,NaN,NaN,...,1.000000,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.241863
3,siglip_rooms,random_baseline,multi_object,en,1,1,0,0.000000,NaN,NaN,...,1.000000,NaN,NaN,0.333333,NaN,NaN,NaN,NaN,NaN,0.199539
4,siglip_rooms,random_baseline,multi_object,es,1,1,0,0.000000,NaN,NaN,...,1.000000,NaN,NaN,0.500000,NaN,NaN,NaN,NaN,NaN,0.384970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,sunrgbd,siglip_v2,negative,es,4,0,4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,29.334388
80,sunrgbd,siglip_v2,object,en,6,6,0,0.833333,0.500000,1.000000,...,1.000000,1.000000,1.0,0.916667,0.750000,1.000000,NaN,NaN,NaN,30.348270
81,sunrgbd,siglip_v2,object,es,6,6,0,0.500000,0.166667,0.833333,...,0.833333,0.500000,1.0,0.655556,0.355556,0.916667,NaN,NaN,NaN,30.258907
82,sunrgbd,siglip_v2,room,en,13,13,0,0.538462,0.307692,0.769231,...,1.000000,1.000000,1.0,0.717949,0.551282,0.884615,NaN,NaN,NaN,30.884247


,dataset_id,method,scope,n_queries,mean_words,recall_at_1,recall_at_1_ci_low,recall_at_1_ci_high,recall_at_3,recall_at_3_ci_low,recall_at_3_ci_high,recall_at_5,recall_at_5_ci_low,recall_at_5_ci_high,reciprocal_rank,reciprocal_rank_ci_low,reciprocal_rank_ci_high
0,siglip_rooms,siglip_v1,descriptive,4,9.250000,1.00,1.00000,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.000000,1.000000,1.000000
1,siglip_rooms,siglip_v1,general,3,4.333333,1.00,1.00000,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.000000,1.000000,1.000000
2,siglip_rooms,siglip_v1,specific,4,7.000000,1.00,1.00000,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.000000,1.000000,1.000000
3,siglip_rooms,siglip_v2,descriptive,4,9.250000,1.00,1.00000,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.000000,1.000000,1.000000
4,siglip_rooms,siglip_v2,general,3,4.333333,1.00,1.00000,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.000000,1.000000,1.000000
5,siglip_rooms,siglip_v2,specific,4,7.000000,1.00,1.00000,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.000000,1.000000,1.000000
6,sunrgbd,siglip_v1,descriptive,10,9.000000,0.40,0.10000,0.70,0.70,0.40,1.00,0.80,0.50,1.0,0.579444,0.342194,0.800000
7,sunrgbd,siglip_v1,general,20,4.050000,0.70,0.50000,0.90,1.00,1.00,1.00,1.00,1.00,1.0,0.833333,0.708333,0.941667
8,sunrgbd,siglip_v1,specific,20,6.250000,0.60,0.40000,0.80,0.75,0.55,0.95,0.90,0.75,1.0,0.713333,0.554979,0.866667
9,sunrgbd,siglip_v2,descriptive,10,9.000000,0.30,0.00000,0.60,0.70,0.40,1.00,0.80,0.50,1.0,0.524359,0.314567,0.733333


,dataset_id,n_paired_queries,delta_recall_at_1,delta_recall_at_1_ci_low,delta_recall_at_1_ci_high,delta_reciprocal_rank,delta_reciprocal_rank_ci_low,delta_reciprocal_rank_ci_high
0,siglip_rooms,11,0.00,0.00,0.0,0.000000,0.000000,0.000000
1,sunrgbd,50,-0.04,-0.18,0.1,-0.020462,-0.108148,0.068097


,dataset_id,method,descriptive_threshold,balanced_accuracy,positive_acceptance,negative_rejection,n_positive,n_negative,warning
0,siglip_rooms,siglip_v1,0.054272,1.0000,1.00,1.000,11,2,descriptive only; estimated and evaluated on t...
1,siglip_rooms,siglip_v2,0.100854,1.0000,1.00,1.000,11,2,descriptive only; estimated and evaluated on t...
2,sunrgbd,siglip_v1,0.073337,0.9375,1.00,0.875,50,8,descriptive only; estimated and evaluated on t...
3,sunrgbd,siglip_v2,0.121123,0.9900,0.98,1.000,50,8,descriptive only; estimated and evaluated on t...


,method,model_id,model_load_s,device,parameters,parameter_memory_mb,embedding_dimension,mean_image_encoding_ms,std_image_encoding_ms,mean_text_encoding_ms,std_text_encoding_ms,benchmark_samples
0,siglip_v1,google/siglip-base-patch16-224,0.699034,cuda,203155970,774.978523,768,19.658274,1.673367,18.744306,43.484422,20
1,siglip_v2,google/siglip2-base-patch16-224,4.975223,cuda,375187970,1431.228523,768,18.386193,0.562547,8.269996,0.283473,20


## Figuras

Se exportan en PNG para inspección y en PDF vectorial para su incorporación a la memoria.

In [3]:
from reproducibility import collect_manifest, save_manifest
results_root = resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'vlm_comparison'
figures_root = results_root / 'figures'
results_root.mkdir(parents=True, exist_ok=True)
for name, frame in results.items():
    frame.to_csv(results_root / f'{name}.csv', index=False)
figure_paths = create_vlm_figures(cases, results['model_costs'], figures_root)
manifest = collect_manifest(ctx['config'], repo_dir=str(ctx['repo_root']), device=ctx['device'],
    extra={'notebook': '01_siglip_retrieval', 'n_cases': len(cases),
           'models': ctx['config']['models']['siglip']['variants'],
           'figures': figure_paths})
save_manifest(str(results_root / 'manifest.json'), manifest)
print(f'Resultados: {results_root}')
print('Figuras generadas:')
for path in figure_paths:
    print(' -', path)

Resultados: /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison
Figuras generadas:
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_por_tipo.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_por_tipo.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_at_k.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_at_k.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_comparacion_idiomas.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_comparacion_idiomas.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/vlm_comparison/figures/vlm_recall_por_alcance_consulta.png
 - /home/junior/visual_semantic_navigation/exper

## Lectura automática de control

In [4]:
sun = cases.loc[(cases['dataset_id'] == 'sunrgbd') & (~cases['is_negative'])
                & cases['method'].isin(['siglip_v1', 'siglip_v2'])]
control = sun.groupby('method')[['recall_at_1', 'recall_at_3', 'recall_at_5', 'reciprocal_rank']].mean()
display(control)
winner = control['recall_at_1'].idxmax()
print(f'Mayor Recall@1 descriptivo sobre SUN RGB-D: {winner} ({control.loc[winner, "recall_at_1"]:.3f}).')
print('La selección definitiva requiere considerar también coste e intervalos, no solo esta media.')

,recall_at_1,recall_at_3,recall_at_5,reciprocal_rank
method,,,,
siglip_v1,0.6,0.84,0.92,0.734556
siglip_v2,0.56,0.86,0.9,0.714094


Mayor Recall@1 descriptivo sobre SUN RGB-D: siglip_v1 (0.600).
La selección definitiva requiere considerar también coste e intervalos, no solo esta media.
